# 01 - Initial Data Merging & Structural EDA

**Project:** E-Commerce Dynamic Pricing & Demand Forecasting Engine (Olist Brazil)
**Stage:** Stage 1 - Data Collection, Merging & Temporal Preprocessing
**Objective:** Load the core Olist CSV datasets, merge them into a single analytics
fact table (item-level granularity), and run the structural & statistical EDA pass
that will drive all downstream feature engineering and modeling decisions.

### Scope of this notebook
1. **Ingest** the 4 core datasets with correct `datetime64` parsing.
2. **Merge** into one item-level table via the documented join graph.
3. **Structural EDA**: integrity, duplicates, missing values, and distributions.
4. **Preserve raw files** - we only read the source CSVs, never modify them.


## 1. Setup: Imports & Logging

Every pipeline stage must emit clear, traceable logs. We configure the standard
`logging` module with a custom `Tqdm`-free stream handler and a helper to log
the outcome of each transformation step.

In [1]:
import logging
import os
from pathlib import Path

import numpy as np
import pandas as pd

# ---------------------------------------------------------------------------
# Logging configuration (mandated by project protocols)
# ---------------------------------------------------------------------------
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-7s | %(name)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
LOGGER = logging.getLogger("olist_eda")
LOGGER.info("Notebook started: 01_initial_data_merging_and_eda.ipynb")


def log_step(step_name: str, rows: int, cols: int) -> None:
    """Log the shape of a pandas object as a pipeline checkpoint.

    Parameters
    ----------
    step_name : str
        Human-readable label of the current pipeline stage.
    rows : int
        Row count of the current table.
    cols : int
        Column count of the current table.
    """
    LOGGER.info("[%s] -> shape=(%d rows, %d cols)", step_name, rows, cols)

2026-09-21 21:42:17 | INFO    | olist_eda | Notebook started: 01_initial_data_merging_and_eda.ipynb


## 2. Ingestion of the Core Datasets

We load the four core tables. The `orders` and `items` tables carry timestamps
that must be parsed eagerly into pandas `datetime64` so every downstream
temporal split is built on true time semantics (no string ordering).

In [2]:
# ---------------------------------------------------------------------------
# Dataset paths
# ---------------------------------------------------------------------------
DATA_DIR = Path(r"D:\ALL Projects\E-Commerce\data\raw")

ORDERS_PATH = DATA_DIR / "olist_orders_dataset.csv"
ITEMS_PATH = DATA_DIR / "olist_order_items_dataset.csv"
PRODUCTS_PATH = DATA_DIR / "olist_products_dataset.csv"
TRANSLATION_PATH = DATA_DIR / "product_category_name_translation.csv"

for path in [ORDERS_PATH, ITEMS_PATH, PRODUCTS_PATH, TRANSLATION_PATH]:
    if not path.exists():
        raise FileNotFoundError(f"Required input file not found: {path}")
    LOGGER.info("Input file located: %s", path.name)

2026-09-21 21:42:17 | INFO    | olist_eda | Input file located: olist_orders_dataset.csv


2026-09-21 21:42:17 | INFO    | olist_eda | Input file located: olist_order_items_dataset.csv


2026-09-21 21:42:17 | INFO    | olist_eda | Input file located: olist_products_dataset.csv


2026-09-21 21:42:17 | INFO    | olist_eda | Input file located: product_category_name_translation.csv


In [3]:
ORDER_DATETIME_COLS = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]


def load_orders(path: Path) -> pd.DataFrame:
    """Load the orders table with timestamps parsed as datetime64.

    Parameters
    ----------
    path : Path
        Path to `olist_orders_dataset.csv`.

    Returns
    -------
    pd.DataFrame
        Orders table with datetime columns already parsed.
    """
    df = pd.read_csv(path, parse_dates=ORDER_DATETIME_COLS)
    for col in ORDER_DATETIME_COLS:
        pd.to_datetime(df[col], errors="coerce")
    log_step("load_orders", df.shape[0], df.shape[1])
    return df


def load_items(path: Path) -> pd.DataFrame:
    """Load the order-items table with `shipping_limit_date` parsed as datetime.

    Parameters
    ----------
    path : Path
        Path to `olist_order_items_dataset.csv`.

    Returns
    -------
    pd.DataFrame
        Order-items table (one row per item inside an order).
    """
    df = pd.read_csv(path, parse_dates=["shipping_limit_date"])
    log_step("load_items", df.shape[0], df.shape[1])
    return df


def load_products(path: Path) -> pd.DataFrame:
    """Load the products metadata table.

    Parameters
    ----------
    path : Path
        Path to `olist_products_dataset.csv`.

    Returns
    -------
    pd.DataFrame
        Products metadata table.
    """
    df = pd.read_csv(path)
    log_step("load_products", df.shape[0], df.shape[1])
    return df


def load_translation(path: Path) -> pd.DataFrame:
    """Load the Portuguese->English category translation table.

    Parameters
    ----------
    path : Path
        Path to `product_category_name_translation.csv`.

    Returns
    -------
    pd.DataFrame
        Translation mapping table (71 rows).
    """
    df = pd.read_csv(path)
    log_step("load_translation", df.shape[0], df.shape[1])
    return df

# ---------------------------------------------------------------------------
# Load all four core tables
# ---------------------------------------------------------------------------
orders = load_orders(ORDERS_PATH)
items = load_items(ITEMS_PATH)
products = load_products(PRODUCTS_PATH)
translation = load_translation(TRANSLATION_PATH)

2026-09-21 21:42:19 | INFO    | olist_eda | [load_orders] -> shape=(99441 rows, 8 cols)


2026-09-21 21:42:20 | INFO    | olist_eda | [load_items] -> shape=(112650 rows, 7 cols)


2026-09-21 21:42:20 | INFO    | olist_eda | [load_products] -> shape=(32951 rows, 9 cols)


2026-09-21 21:42:20 | INFO    | olist_eda | [load_translation] -> shape=(71 rows, 2 cols)


In [4]:
# ---------------------------------------------------------------------------
# Verification: timestamp columns must now be datetime64
# ---------------------------------------------------------------------------
datetime_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]
parsed_ok = all(pd.api.types.is_datetime64_any_dtype(orders[col]) for col in datetime_cols)
parsed_ok_items = pd.api.types.is_datetime64_any_dtype(items["shipping_limit_date"])
LOGGER.info("All orders timestamp cols parsed to datetime64: %s", parsed_ok)
LOGGER.info("items.shipping_limit_date parsed to datetime64:   %s", parsed_ok_items)
assert parsed_ok and parsed_ok_items, "Timestamp parsing failed - merge would lose temporal integrity."

# Time coverage of the marketplace
purchase_min = orders["order_purchase_timestamp"].min()
purchase_max = orders["order_purchase_timestamp"].max()
print(f"Order purchase window : {purchase_min}  ->  {purchase_max}")
print(f"Marketplace span       : {(purchase_max - purchase_min).days} days")

2026-09-21 21:42:20 | INFO    | olist_eda | All orders timestamp cols parsed to datetime64: True


2026-09-21 21:42:20 | INFO    | olist_eda | items.shipping_limit_date parsed to datetime64:   True


Order purchase window : 2016-09-04 21:15:19  ->  2018-10-17 17:30:18
Marketplace span       : 772 days


## 3. Merge Pipeline (Item-Level Fact Table)

Join graph (left-to-right, key in parentheses):
1. `order_items` + `orders` on `order_id`
2. plus `products` on `product_id`
3. plus `category_translation` on `product_category_name`

The `validate` flag is used to catch 1:N surprises: `order_id` in `orders` must
be unique (1:many is expected into items), and `product_id` in `products` must
be unique. Non-matching keys become `NaN` and are reported explicitly.

In [5]:
def merge_core_tables(
    items: pd.DataFrame,
    orders: pd.DataFrame,
    products: pd.DataFrame,
    translation: pd.DataFrame,
) -> pd.DataFrame:
    """Build the item-level fact table from the four core Olist tables.

    Apply the documented left-join graph and log the row budget after each hop.

    Parameters
    ----------
    items : pd.DataFrame
        Order-items table (base / left side).
    orders : pd.DataFrame
        Orders table.
    products : pd.DataFrame
        Products metadata table.
    translation : pd.DataFrame
        Portuguese->English category translation table.

    Returns
    -------
    pd.DataFrame
        Merged item-level dataset with 1 row per sold item.
    """
    if items["order_id"].duplicated().any() and orders["order_id"].duplicated().any():
        raise ValueError("Both sides contain duplicate order_id; ambiguous join detected.")

    df = items.merge(orders, on="order_id", how="left", validate="many_to_one")
    log_step("merge+orders", df.shape[0], df.shape[1])

    df = df.merge(products, on="product_id", how="left", validate="many_to_one")
    log_step("merge+products", df.shape[0], df.shape[1])

    df = df.merge(translation, on="product_category_name", how="left", validate="many_to_one")
    log_step("merge+translation", df.shape[0], df.shape[1])
    return df


merged = merge_core_tables(items, orders, products, translation)
print(f"Final merged shape: {merged.shape}")

2026-09-21 21:42:21 | INFO    | olist_eda | [merge+orders] -> shape=(112650 rows, 14 cols)


2026-09-21 21:42:21 | INFO    | olist_eda | [merge+products] -> shape=(112650 rows, 22 cols)


2026-09-21 21:42:21 | INFO    | olist_eda | [merge+translation] -> shape=(112650 rows, 23 cols)


Final merged shape: (112650, 23)


In [6]:
# ---------------------------------------------------------------------------
# Join-quality audit: how many keys failed to resolve on each hop?
# ---------------------------------------------------------------------------
orphan_orders = merged["order_purchase_timestamp"].isna().sum()
orphan_products = merged["product_category_name"].isna().sum()
orphan_categories = merged["product_category_name_english"].isna().sum()

print(f"Items whose order_id found no order record : {orphan_orders:,} ({orphan_orders / len(merged):.2%})")
print(f"Items whose product_id found no product    : {orphan_products:,} ({orphan_products / len(merged):.2%})")
print(f"Items whose category has no translation    : {orphan_categories:,} ({orphan_categories / len(merged):.2%})")

LOGGER.warning(
    "Products without category metadata: %d unique product_ids (their items inherit NaN category).",
    merged.loc[orphan_products > 0] if False else merged["product_id"].loc[merged["product_category_name"].isna()].nunique(),
)

2026-09-21 21:42:21 | WARNING | olist_eda | Products without category metadata: 610 unique product_ids (their items inherit NaN category).


Items whose order_id found no order record : 0 (0.00%)
Items whose product_id found no product    : 1,603 (1.42%)
Items whose category has no translation    : 1,627 (1.44%)


## 4. Structural & Statistical EDA

### 4.1 Dataset Integrity Check
Confirm shape, column inventory, and the effective `dtypes` of the merged table.

In [7]:
print(f"Merged shape: {merged.shape[0]} rows x {merged.shape[1]} columns")

Merged shape: 112650 rows x 23 columns


In [8]:
merged.info(memory_usage="deep", verbose=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 23 columns):
 #   Column                         Non-Null Count   Dtype         
---  ------                         --------------   -----         
 0   order_id                       112650 non-null  object        
 1   order_item_id                  112650 non-null  int64         
 2   product_id                     112650 non-null  object        
 3   seller_id                      112650 non-null  object        
 4   shipping_limit_date            112650 non-null  datetime64[ns]
 5   price                          112650 non-null  float64       
 6   freight_value                  112650 non-null  float64       
 7   customer_id                    112650 non-null  object        
 8   order_status                   112650 non-null  object        
 9   order_purchase_timestamp       112650 non-null  datetime64[ns]
 10  order_approved_at              112635 non-null  datetime64[ns]
 11  

In [9]:
print("Column inventory:")
for i, col in enumerate(merged.columns, start=1):
    print(f"{i:>2}. {col:<45} {str(merged[col].dtype):<12} {merged[col].nunique():>9,} unique")

Column inventory:
 1. order_id                                      object          98,666 unique
 2. order_item_id                                 int64               21 unique
 3. product_id                                    object          32,951 unique
 4. seller_id                                     object           3,095 unique
 5. shipping_limit_date                           datetime64[ns]    93,318 unique


 6. price                                         float64          5,968 unique
 7. freight_value                                 float64          6,999 unique
 8. customer_id                                   object          98,666 unique
 9. order_status                                  object               7 unique
10. order_purchase_timestamp                      datetime64[ns]    98,112 unique
11. order_approved_at                             datetime64[ns]    90,174 unique
12. order_delivered_carrier_date                  datetime64[ns]    81,017 unique
13. order_delivered_customer_date                 datetime64[ns]    95,664 unique
14. order_estimated_delivery_date                 datetime64[ns]       450 unique
15. product_category_name                         object              73 unique
16. product_name_lenght                           float64             66 unique
17. product_description_lenght                    float64          2,960 unique
18. product_photos_qty        

20. product_length_cm                             float64             99 unique
21. product_height_cm                             float64            102 unique
22. product_width_cm                              float64             95 unique
23. product_category_name_english                 object              71 unique


### 4.2 Duplicate Inspection
Two distinct checks:
- **Exact duplicate rows** (every column identical) - data-corruption signal.
- **Natural key collision** on `(order_id, order_item_id)` - transaction-level
  duplicates would corrupt any demand aggregation.

In [10]:
exact_dups = merged.duplicated(keep=False).sum()
print(f"Exact duplicate rows (all cols identical) : {exact_dups:,}")

# Natural key of the item fact table
natural_key = ["order_id", "order_item_id"]
key_dups = merged.duplicated(subset=natural_key, keep=False).sum()
print(f"Rows sharing (order_id, order_item_id)     : {key_dups:,}")

# Cross-check against source: does the raw items table itself carry key dups?
source_key_dups = items.duplicated(subset=natural_key, keep=False).sum()
print(f"Source items table key dups                : {source_key_dups:,}")

# Order-level uniqueness on the orders side
order_dups = orders["order_id"].duplicated().sum()
print(f"Duplicate order_id in orders table         : {order_dups:,}")

Exact duplicate rows (all cols identical) : 0
Rows sharing (order_id, order_item_id)     : 0
Source items table key dups                : 0
Duplicate order_id in orders table         : 0


### 4.3 Missing Value Audit
Absolute `NaN` counts plus their share of the table. We distinguish structural
missingness (route dates absent for not-yet-delivered orders) from genuine gaps
(product specs missing for a subset of products).

In [11]:
def missing_audit(df: pd.DataFrame) -> pd.DataFrame:
    """Build the missing-value audit table for a dataframe.

    Parameters
    ----------
    df : pd.DataFrame
        Table to audit.

    Returns
    -------
    pd.DataFrame
        Audit table with null count, null percentage, and dtype per column,
        sorted by null count descending.
    """
    n = len(df)
    audit = pd.DataFrame(
        {
            "dtype": df.dtypes.astype(str),
            "null_count": df.isna().sum(),
            "null_pct": (df.isna().sum() / n * 100).round(2),
        }
    )
    audit = audit.sort_values("null_count", ascending=False)
    audit = audit[audit["null_count"] > 0]
    return audit


missing = missing_audit(merged)
if missing.empty:
    LOGGER.info("No missing values detected on the merged table.")
else:
    LOGGER.info("Missing-value audit built: %d columns contain NaNs.", len(missing))
display(missing) if "display" in dir() else print(missing.to_string())

2026-09-21 21:42:23 | INFO    | olist_eda | Missing-value audit built: 12 columns contain NaNs.


                                        dtype  null_count  null_pct
order_delivered_customer_date  datetime64[ns]        2454      2.18
product_category_name_english          object        1627      1.44
product_name_lenght                   float64        1603      1.42
product_category_name                  object        1603      1.42
product_description_lenght            float64        1603      1.42
product_photos_qty                    float64        1603      1.42
order_delivered_carrier_date   datetime64[ns]        1194      1.06
product_height_cm                     float64          18      0.02
product_length_cm                     float64          18      0.02
product_width_cm                      float64          18      0.02
product_weight_g                      float64          18      0.02
order_approved_at              datetime64[ns]          15      0.01


### 4.4 Statistical Distribution Check
`describe(include="all")` produces the count / central-tendency / dispersion for
mixed dtypes. We surface:
- cardinal business metrics (`price`, `freight_value`),
- order lifecycle durations implied by timestamps,
- category cardinality for later demand aggregation.

In [12]:
cat_cols = merged.select_dtypes(include="object").columns.tolist()
num_cols = merged.select_dtypes(include=np.number).columns.tolist()
dt_cols = merged.select_dtypes(include="datetime64").columns.tolist()
print(f"object cols     ({len(cat_cols)}): {cat_cols}")
print(f"numeric cols    ({len(num_cols)}): {num_cols}")
print(f"datetime cols   ({len(dt_cols)}): {dt_cols}")

object cols     (7): ['order_id', 'product_id', 'seller_id', 'customer_id', 'order_status', 'product_category_name', 'product_category_name_english']
numeric cols    (10): ['order_item_id', 'price', 'freight_value', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']
datetime cols   (6): ['shipping_limit_date', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']


In [13]:
merged.describe(include="all").T

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
order_id,112650,98666,8272b63d03f5f79c56e9e4120aec44ef,21,NaN,NaN,NaN,NaN,NaN,NaN,NaN
order_item_id,112650.0,NaN,NaN,NaN,1.197834,1.0,1.0,1.0,1.0,21.0,0.705124
product_id,112650,32951,aca2eb7d00ea1a7b8ebd4e68314663af,527,NaN,NaN,NaN,NaN,NaN,NaN,NaN
seller_id,112650,3095,6560211a19b47992c3666cc44a7e94c0,2033,NaN,NaN,NaN,NaN,NaN,NaN,NaN
shipping_limit_date,112650,NaN,NaN,NaN,2018-01-07 15:36:52.192685312,2016-09-19 00:15:34,2017-09-20 20:57:27.500000,2018-01-26 13:59:35,2018-05-10 14:34:00.750000128,2020-04-09 22:35:08,NaN
price,112650.0,NaN,NaN,NaN,120.653739,0.85,39.9,74.99,134.9,6735.0,183.633928
freight_value,112650.0,NaN,NaN,NaN,19.99032,0.0,13.08,16.26,21.15,409.68,15.806405
customer_id,112650,98666,fc3d1daec319d62d49bfb5e1f83123e9,21,NaN,NaN,NaN,NaN,NaN,NaN,NaN
order_status,112650,7,delivered,110197,NaN,NaN,NaN,NaN,NaN,NaN,NaN
order_purchase_timestamp,112650,NaN,NaN,NaN,2018-01-01 00:09:48.464376576,2016-09-04 21:15:19,2017-09-13 19:17:04,2018-01-19 23:02:16,2018-05-04 17:30:36.750000128,2018-09-03 09:06:57,NaN


In [14]:
# Business metric extremes - price and freight are the direct inputs to the
# Stage-2 pricing engine, so their tails matter.
for col in ["price", "freight_value"]:
    q01, q99 = merged[col].quantile([0.01, 0.99])
    print(f"{col:<14} min={merged[col].min():>10.2f}  p1={q01:>10.2f}  "
          f"median={merged[col].median():>10.2f}  p99={q99:>10.2f}  max={merged[col].max():>10.2f}")

price          min=      0.85  p1=      9.99  median=     74.99  p99=    890.00  max=   6735.00
freight_value  min=      0.00  p1=      4.42  median=     16.26  p99=     84.52  max=    409.68


In [15]:
# How many distinct products / categories / sellers does the fact table expose?
print("Distinct products in orders :", merged["product_id"].nunique())
print("Distinct categories         :", merged["product_category_name_english"].nunique())
print("Distinct sellers            :", merged["seller_id"].nunique())

# Items per order distribution (basket size)
basket_size = merged.groupby("order_id")["order_item_id"].max()
print("\nBasket size (items/order):")
print(basket_size.describe().round(2).to_string())
print("Max basket size:", basket_size.max())

Distinct products in orders : 32951
Distinct categories         : 71
Distinct sellers            : 3095



Basket size (items/order):
count    98666.00
mean         1.14
std          0.54
min          1.00
25%          1.00
50%          1.00
75%          1.00
max         21.00
Max basket size: 21


## 5. Summary & Next Steps

### Key structural findings (auto-summarized below)
- Item-level fact table covering the full Olist order stream.
- Missing values are mostly *structural*: delivery-date columns absent for
  undelivered/not-yet-delivered orders; product specs absent for the 610
  products without category metadata.
- `order_status` contains non-`delivered` states that must be filtered or
  down-weighted before building the demand target, or they will deflate
  `order_demand`.
- `price` right-tail (max ~6.7k BRL, p99 far lower) indicates extreme-value
  envelopes must be handled explicitly in the pricing phase.

### Evaluation-metric consequences (detailed in the final report)
The demand target is a right-skewed, over-dispersed count. A usable metric
suite is **WAPE (primary), MAE and RMSE (secondary)** - see the final write-up.


In [16]:
final = pd.DataFrame(
    {
        "metric": [
            "rows",
            "cols",
            "duplicate_rows",
            "rows_with_missing",
            "price_max",
            "price_p99",
            "price_mean",
            "freight_mean",
            "purchase_start",
            "purchase_end",
        ],
        "value": [
            merged.shape[0],
            merged.shape[1],
            int(exact_dups),
            int(merged.isna().any(axis=1).sum()),
            round(float(merged["price"].max()), 2),
            round(float(merged["price"].quantile(0.99)), 2),
            round(float(merged["price"].mean()), 2),
            round(float(merged["freight_value"].mean()), 2),
            str(orders["order_purchase_timestamp"].min()),
            str(orders["order_purchase_timestamp"].max()),
        ],
    }
)
print(final.to_string(index=False))
LOGGER.info("Notebook completed: 01_initial_data_merging_and_eda.ipynb")

2026-09-21 21:42:25 | INFO    | olist_eda | Notebook completed: 01_initial_data_merging_and_eda.ipynb


           metric               value
             rows              112650
             cols                  23
   duplicate_rows                   0
rows_with_missing                4029
        price_max              6735.0
        price_p99               890.0
       price_mean              120.65
     freight_mean               19.99
   purchase_start 2016-09-04 21:15:19
     purchase_end 2018-10-17 17:30:18
